Анализ деятельности кофейной сети на основе данных о продажах, клиентах и продукции.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Загрузка данных
sales = pd.read_csv('Coffee Shop Sales.csv')
customers = pd.read_csv('customers.csv')

In [2]:
#выводим все столбцы
print(sales.columns)
print(customers.columns)

Index(['transaction_id', 'transaction_date', 'transaction_time',
       'transaction_qty', 'store_id', 'store_location', 'product_id',
       'unit_price', 'product_category', 'product_type', 'product_detail'],
      dtype='object')
Index(['customer id', 'customer name', 'email', 'phone number',
       'address line 1', 'postcode', 'loyalty card', 'location',
       'date of birth'],
      dtype='object')


In [3]:
#Выводим все данные в виде таблицы

sales.head()



,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,01-01-2023,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,01-01-2023,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,01-01-2023,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,01-01-2023,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,01-01-2023,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [3]:
customers.head()

,customer id,customer name,email,phone number,address line 1,postcode,loyalty card,location,date of birth
0,17670-51384-MA,Aloisia Allner,aallner0@lulu.com,+1 (862) 817-0124,57999 Pepper Wood Alley,7505,Yes,Astoria,1970-04-21
1,73342-18763-UW,Piotr Bote,pbote1@yelp.com,+353 (913) 396-4653,2112 Ridgeway Hill,D6W,No,Lower Manhattan,1974-04-11
2,21125-22134-PX,Jami Redholes,jredholes2@tmall.com,+1 (210) 986-6806,5214 Bartillon Park,78205,Yes,Hell's Kitchen,2006-06-19
3,71253-00052-RN,Dene Azema,dazema3@facebook.com,+1 (217) 418-0714,27 Maywood Place,62711,Yes,Hell's Kitchen,1974-04-22
4,23806-46781-OU,Christoffer O' Shea,NaN,+353 (698) 362-9201,38980 Manitowish Junction,N41,No,Hell's Kitchen,1998-07-04


In [5]:
# Расчет выручки
sales['revenue'] = sales['unit_price'] * sales['transaction_qty']
sales[['transaction_date','product_category','revenue']].head()


,transaction_date,product_category,revenue
0,01-01-2023,Coffee,6.0
1,01-01-2023,Tea,6.2
2,01-01-2023,Drinking Chocolate,9.0
3,01-01-2023,Coffee,2.0
4,01-01-2023,Tea,6.2


In [6]:
# Базовые метрики
total_transactions = sales['transaction_id'].nunique()
total_revenue = sales['revenue'].sum().round(2)
average_check = sales.groupby('transaction_id')['revenue'].sum().mean().round(2)


print(f"Общее количество транзакций: {total_transactions}")
print(f"Общая выручка: {total_revenue}")
print(f"Средний чек: {average_check}")

Общее количество транзакций: 149116
Общая выручка: 698812.33
Средний чек: 4.69


In [ ]:
#создание столбцов год, месяц, день, час
sales['transaction_date']=pd.to_datetime(sales['transaction_date'], format='%d-%m-%Y', dayfirst=True)
sales['year']=sales['transaction_date'].dt.year
sales['month']=sales['transaction_date'].dt.month_name()
sales['day_of_week']=sales['transaction_date'].dt.day_name()

sales['transaction_time'] = pd.to_datetime(sales['transaction_time'], errors='coerce')
sales['hour'] = sales['transaction_time'].dt.hour 
sales['transaction_time'] = sales['transaction_time'].dt.strftime('%H:%M:%S')

C:\Users\Artem\AppData\Local\Temp\ipykernel_15880\3202646500.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sales['transaction_time'] = pd.to_datetime(sales['transaction_time'], errors='coerce')


In [17]:
sales.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,revenue,year,month,day,day_of_week,hour
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.0,2023,January,1,Sunday,7
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,2023,January,1,Sunday,7
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.0,2023,January,1,Sunday,7
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.0,2023,January,1,Sunday,7
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,2023,January,1,Sunday,7


In [18]:
#Сохраняем обработанный файл, для построения графиков в Tableau

sales.to_csv('Sales.csv')


In [ ]:
from datetime import datetime


# Преобразуем даты рождения
customers['date of birth'] = pd.to_datetime(customers['date of birth'])

# Рассчитываем возраст
current_year = datetime.now().year
customers['age'] = current_year - customers['date of birth'].dt.year

# Создаем новую колонку с категоризацией по возрасту
def categorize_age(age):
    if age < 25:
        return 'Молодежь'
    elif 25 <= age <= 44:
        return 'Средний возраст'
    else:
        return 'Пожилой возраст'

# Применяем функцию к каждому значению возраста
customers['age_group'] = customers['age'].apply(categorize_age)


# Сохраняем обновленный датафрейм
customers.to_csv('customers.csv', index=False)


